# Import Libraries and Parameters

In [24]:
%load_ext autoreload
%autoreload 2
%matplotlib qt6

import dataclasses
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as patches
from matplotlib.ticker import NullLocator
import scipy as sp
import pandas as pd
import Collected_runs as cr
import parameters
import FS3
import ctypes
import subprocess

mpl.rcParams.update(
    {
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": "cmr10",
        "mathtext.fontset": "cm",
        "font.family": "STIXGeneral",
        "axes.unicode_minus": True,
        "axes.labelsize": 21,
        "xtick.labelsize": 18,
        "ytick.labelsize": 18,
        "legend.fontsize": 16,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.top": "on",
        "xtick.major.bottom": "on",
        "ytick.major.left": "on",
        "ytick.major.right": "on",
        "xtick.top": True,
        "ytick.right": True,
        "figure.autolayout": True,
    }
)

def rand_start_params(start_params, fitFunc):
    """start params = [gc, nu, c, omega, a0,a1,a2,a3]"""
    params_rand = np.random.randn(fitFunc.nparams() - len(start_params)) * 0.0001
    return (
        np.concatenate((start_params, params_rand))
        + np.random.randn(fitFunc.nparams()) * 0.0001
    )

def correct_hist_df(fitFunc, hist_df):
    mask = hist_df["omega"] < 0

    hist_df.loc[mask, "z"] = hist_df.loc[mask, "omega"] + hist_df.loc[mask, "z"]
    hist_df.loc[mask, "dz"] = np.sqrt(
        hist_df.loc[mask, "dz"]**2 + hist_df.loc[mask, "domega"]**2
    )

    hist_df.loc[mask, "omega"] = -hist_df.loc[mask, "omega"]

    # polynomial coefficients
    for n in range(1, fitFunc.polyOrder + 1):
        a_col = f"a{n}"
        da_col = f"da{n}"

        hist_df.loc[mask, a_col] = (
            hist_df.loc[mask, "c"] * hist_df.loc[mask, a_col]
        )

        hist_df.loc[mask, da_col] = np.sqrt(
            (hist_df.loc[mask, a_col] * hist_df.loc[mask, "dc"])**2 +
            (hist_df.loc[mask, "c"] * hist_df.loc[mask, da_col])**2
        )

    hist_df.loc[mask, "c"] = 1 / hist_df.loc[mask, "c"]
    hist_df.loc[mask, "dc"] = hist_df.loc[mask, "dc"] * hist_df.loc[mask, "c"]**2

    print("corrected for negative omega")
    return hist_df

_EWALD_SRC = "J_ewald.cpp"
_EWALD_LIB = "J_ewald.so"

# Recompile only if the source is newer than the compiled library
if not os.path.exists(_EWALD_LIB) or os.path.getmtime(_EWALD_SRC) > os.path.getmtime(_EWALD_LIB):
    subprocess.run(["g++", "-O3", "-shared", "-fPIC", _EWALD_SRC, "-o", _EWALD_LIB], check=True)

_ewald_lib = ctypes.CDLL(os.path.abspath(_EWALD_LIB))
_ewald_lib.ewald_summation_1D.argtypes = [
    ctypes.c_size_t,  # L
    ctypes.c_int,     # d
    ctypes.c_double,  # alpha
    ctypes.c_int,     # n_cutoff
]
_ewald_lib.ewald_summation_2D.argtypes = [
    ctypes.c_size_t,  # Lx
    ctypes.c_size_t,  # Ly
    ctypes.c_int,     # dx
    ctypes.c_int,     # dy
    ctypes.c_double,  # alpha
    ctypes.c_int,     # n_cutoff
]
_ewald_lib.ewald_summation_1D.restype = ctypes.c_double
_ewald_lib.ewald_summation_2D.restype = ctypes.c_double


def ewald_summation_2D(L, dx, dy, alpha, n_cutoff):
    return _ewald_lib.ewald_summation_2D(int(L), int(L), int(dx), int(dy), float(alpha), int(n_cutoff))

def ewald_summation_1D_zeta(L,dx,alpha):
    return L**(-alpha) * (sp.special.zeta(alpha, dx / L) + sp.special.zeta(alpha, 1 - dx / L))

def ewald_summation_1D_cutoff(L,dx,alpha,n_cutoff):
    return _ewald_lib.ewald_summation_1D(int(L), int(dx), float(alpha), int(n_cutoff))

def hopping_SLAC_fermions(L, x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    fx = np.where(x == 0, 0, 1j * (-1)**x * np.pi / (L * np.sin(np.pi * x / L)))
    fy = np.where(y == 0, 0, 1j * (-1)**y * np.pi / (L * np.sin(np.pi * y / L)))

    return fx[..., None] * (y == 0) + 1j*fy[None, ...] * (x == 0)[..., None]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Write for FS3 Data

In [9]:
T0 = False

if T0:
    params = parameters.SELECTPARAMS_J
    name="ZeroTemp"
else:
    params = parameters.SELECTPARAMS_T
    name="FiniteTemp"

for i in range(len(params.alpha_list)):
    alpha = params.alpha_list[i]
    path = params.pathRuns[i]
    cl_runs = cr.collected_runs(path, T0, cutoff=100)

    cl_runs.write_to_FS3_file("/home/daniel/Master_thesis/DataFS3/%s/alpha%.2f/" % (name, alpha))

# Initial observations (TO DO: ALL)

## Check if T=0 (Done)

### $m^2$

In [49]:
plt.close("all")

path = [f"/home/daniel/Master_thesis/Data/LocalData/runs_BETAJ5"]

cl_runs = cr.collected_runs(path, False, cutoff=3)

data, _, _ = cl_runs.get_x_vs_y_data(
    xname="g_param",
    yname="mag2",
    xfunc="1/x",
    yfunc="y/(Ns**2)",
    plot=False,
    xscale="log",
    yscale="log",
    xlabel=r"$^{-1}$",
)

fig, ax0 = FS3.figure(xlabel=r"$\beta$", ylabel=r"$\Delta m_s^2$")
FS3.setTicks(ax0, minorxTicks=2, minoryTicks=2)
FS3.setStyle(Llist=np.sort(list(data), axis=0))

plots = []
labels = []
for L in np.sort(list(data), axis=0):
    plots.append(
        ax0.errorbar(
            data[L][:, 0],
            abs(data[L][:, 1]-data[L][0, 1]),
            data[L][:, 2],
            **FS3.errorbarStyle(L),
            linestyle="-",
        )
    )
    labels.append(r"$L = %s$" % L)

    color = FS3.errorbarStyle(L)["color"]
    i = 7 - int(np.log2(L))
    box = patches.Rectangle((L/1.1,abs(data[L][i, 1]-data[L][0, 1])-1e-5), L*(1.1-1/1.1), 2e-5, edgecolor=color, fc=color, alpha = 0.5)
    ax0.add_patch(box)

ax0.set_xscale("log",base = 2)
ax0.set_yscale("linear")

ax0.set_xlim(3.0, 192)
ax0.tick_params(
    axis="both", labelbottom=True, labeltop=False, labelleft=True, labelright=False
)
plots.append(ax0.hlines(0, 3.0, 192, colors="black", linestyles="--", alpha=0.5))
labels.append("$\Delta m_s^2 = 0$")
ax0.set_xticks([4, 8, 16, 32, 64, 128],labels=[r"$4$", r"$8$", r"$16$", r"$32$", r"$64$", r"$128$"])

FS3.legend(ax0, plots, labels, legendLabelBreak=7, loc=(0.7, 0.7, 0.18))
fig.savefig("Delta_mag2_BETAJ5.pdf")

#########################################################################################################

fig1, ax1 = FS3.figure(xlabel=r"$\beta$", ylabel=r"$m_s^2$")
FS3.setTicks(ax0, minorxTicks=2, minoryTicks=2)

plots = []
labels = []
for L in np.sort(list(data), axis=0):
    plots.append(
        ax1.errorbar(
            data[L][:, 0],
            abs(data[L][:, 1]),
            data[L][:, 2],
            **FS3.errorbarStyle(L),
            linestyle="-",
        )
    )
    labels.append(r"$L = %s$" % L)

    color = FS3.errorbarStyle(L)["color"]
    i = 7 - int(np.log2(L))
    box = patches.Rectangle((L/1.1,abs(data[L][i, 1])-2e-4), L*(1.1-1/1.1), 4e-4, edgecolor=color, fc=color, alpha = 0.5)
    ax1.add_patch(box)

ax1.set_xscale("log",base = 2)
ax1.set_yscale("linear")

ax1.set_xlim(3.0, 192)
ax1.tick_params(
    axis="both", labelbottom=True, labeltop=False, labelleft=True, labelright=False
)
ax1.set_xticks([4, 8, 16, 32, 64, 128],labels=[r"$4$", r"$8$", r"$16$", r"$32$", r"$64$", r"$128$"])
FS3.legend(ax1, plots, labels, legendLabelBreak=7, loc=(0.7, 0.7, 0.18))
fig.savefig("Mag2_BETAJ5.pdf")

<>:49: SyntaxWarning: invalid escape sequence '\D'
<>:49: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_50817/2858342964.py:49: SyntaxWarning: invalid escape sequence '\D'
  labels.append("$\Delta m_s^2 = 0$")
/tmp/ipykernel_50817/2858342964.py:88: UserWarning: AutoMinorLocator does not work on logarithmic scales
  fig.savefig("Mag2_BETAJ5.pdf")


### $U_2$

In [50]:
plt.close("all")

path = [f"/home/daniel/Master_thesis/Data/LocalData/runs_BETAJ5"]

cl_runs = cr.collected_runs(path, False, cutoff=3)

data, _, _ = cl_runs.get_x_vs_y_data(
    xname="g_param",
    yname="binder",
    xfunc="1/x",
    #yfunc="y/(Ns**2)",
    plot=False,
    xscale="log",
    yscale="log",
    xlabel=r"$^{-1}$",
)

fig, ax0 = FS3.figure(xlabel=r"$\beta$", ylabel=r"$\Delta U_2$")
FS3.setTicks(ax0, minorxTicks=2, minoryTicks=2)
FS3.setStyle(Llist=np.sort(list(data), axis=0))

plots = []
labels = []
for L in np.sort(list(data), axis=0):
    plots.append(
        ax0.errorbar(
            data[L][:, 0],
            abs(data[L][:, 1]-data[L][0, 1]),
            data[L][:, 2],
            **FS3.errorbarStyle(L),
            linestyle="-",
        )
    )
    labels.append(r"$L = %s$" % L)

    color = FS3.errorbarStyle(L)["color"]
    i = 7 - int(np.log2(L))
    box = patches.Rectangle((L/1.1,abs(data[L][i, 1]-data[L][0, 1])-0.005), L*(1.1-1/1.1), 0.01, edgecolor=color, fc=color, alpha = 0.5)
    ax0.add_patch(box)

ax0.set_xscale("log",base = 2)
ax0.set_yscale("linear")

ax0.set_xlim(3.0, 192)
ax0.tick_params(
    axis="both", labelbottom=True, labeltop=False, labelleft=True, labelright=False
)
plots.append(ax0.hlines(0, 3.0, 192, colors="black", linestyles="--", alpha=0.5))
labels.append("$\Delta U_2 = 0$")
ax0.set_xticks([4, 8, 16, 32, 64, 128],labels=[r"$4$", r"$8$", r"$16$", r"$32$", r"$64$", r"$128$"])

FS3.legend(ax0, plots, labels, legendLabelBreak=7, loc=(0.7, 0.7, 0.18))
fig.savefig("Delta_Binder_BETAJ5.pdf")

#########################################################################################################

fig1, ax1 = FS3.figure(xlabel=r"$\beta$", ylabel=r"$U_2$")
FS3.setTicks(ax0, minorxTicks=2, minoryTicks=2)

plots = []
labels = []
for L in np.sort(list(data), axis=0):
    plots.append(
        ax1.errorbar(
            data[L][:, 0],
            abs(data[L][:, 1]),
            data[L][:, 2],
            **FS3.errorbarStyle(L),
            linestyle="-",
        )
    )
    labels.append(r"$L = %s$" % L)

    color = FS3.errorbarStyle(L)["color"]
    i = 7 - int(np.log2(L))
    box = patches.Rectangle((L/1.1,abs(data[L][i, 1])-0.005), L*(1.1-1/1.1), 0.01, edgecolor=color, fc=color, alpha = 0.5)
    ax1.add_patch(box)

ax1.set_xscale("log",base = 2)
ax1.set_yscale("linear")

ax1.set_xlim(3.0, 192)
ax1.tick_params(
    axis="both", labelbottom=True, labeltop=False, labelleft=True, labelright=False
)
ax1.set_xticks([4, 8, 16, 32, 64, 128],labels=[r"$4$", r"$8$", r"$16$", r"$32$", r"$64$", r"$128$"])
FS3.legend(ax1, plots, labels, legendLabelBreak=7, loc=(0.7, 0.7, 0.18))
fig.savefig("Binder_BETAJ5.pdf")

<>:49: SyntaxWarning: invalid escape sequence '\D'
<>:49: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_50817/1352062564.py:49: SyntaxWarning: invalid escape sequence '\D'
  labels.append("$\Delta U_2 = 0$")
/tmp/ipykernel_50817/1352062564.py:88: UserWarning: AutoMinorLocator does not work on logarithmic scales
  fig.savefig("Binder_BETAJ5.pdf")


## Extensivity (DONE)

In [99]:
plt.close("all")


def powerlaw(x, a, b):
    return a * x ** (b)


DeltaT = []
errT = []
DeltaJ = []
errJ = []

for T0 in [True, False]:
    if T0:
        params = parameters.SELECTPARAMS_J
        name = "J"
    else:
        params = parameters.SELECTPARAMS_T
        name = "T"

    fig, ax = FS3.figure(xlabel=r"$N$", ylabel=r"$|E|$")
    FS3.setStyle(Llist=[])

    plots = []
    labels = []

    for i in range(len(params.alpha_list)):
        alpha = params.alpha_list[i]
        path = params.pathRuns[i]
        cl_runs = cr.collected_runs(path, T0, cutoff=100)

        data, _, _ = cl_runs.get_x_vs_y_data(
            xname="g_param", yname="energy", plot=False
        )

        g0 = params.gc_estimate[i]

        tab = np.empty((0, 3))
        tab2 = np.empty((0, 3))
        tab3 = np.empty((0, 3))
        for L in np.sort(list(data.keys()), axis=0):
            imin = np.argmin(np.abs(data[L][:, 0] - g0))
            tab = np.vstack((tab, np.array([L, data[L][imin, 1], data[L][imin, 2]])))
            tab2 = np.vstack((tab2, np.array([L, data[L][0, 1], data[L][0, 2]])))
            tab3 = np.vstack((tab3, np.array([L, data[L][-1, 1], data[L][-1, 2]])))

        a_prime = tab[-1, 1] * (tab[-1, 0] ** -2)
        a_prime2 = tab2[-1, 1] * (tab2[-1, 0] ** -2)
        a_prime3 = tab3[-1, 1] * (tab3[-1, 0] ** -2)

        p = ax.errorbar(
            tab[:, 0] ** 2,
            abs((tab[:, 1])),
            tab[:, 2],
            **FS3.errorbarStyle(i),
            ls="",
        )
        # ax.errorbar(
        #     tab2[:, 0] ** 2,
        #     abs((tab2[:, 1] )),
        #     tab2[:, 2],
        #     **FS3.errorbarStyle(i),
        #     ls="",
        # )
        # ax.errorbar(
        #     tab3[:, 0] ** 2,
        #     abs((tab3[:, 1] )),
        #     tab3[:, 2],
        #     **FS3.errorbarStyle(i),
        #     ls="",
        # )
        plots.append(p)
        labels.append(r" $\alpha = %.2f$" % (alpha))

        res, err = sp.optimize.curve_fit(
            powerlaw, tab[:, 0] ** 2, (tab[:, 1]), p0=[a_prime, 1.0]
        )
        res2, err2 = sp.optimize.curve_fit(
            powerlaw, tab2[:, 0] ** 2, (tab2[:, 1]), p0=[a_prime2, 1.0]
        )
        res3, err3 = sp.optimize.curve_fit(
            powerlaw, tab3[:, 0] ** 2, (tab3[:, 1]), p0=[a_prime3, 1.0]
        )

        err = np.sqrt(np.diag(err))
        err2 = np.sqrt(np.diag(err2))
        err3 = np.sqrt(np.diag(err3))

        x = np.linspace((tab[0, 0] ** 2) * 0.95, (tab[-1, 0] ** 2) * 1.05, 100)
        y = powerlaw(x, *res)
        y1 = powerlaw(x, a_prime, 1.0)
        ax.plot(x, abs(y), **FS3.plotStyle(i))
        ax.plot(x, abs(y1), color="black", ls="-", alpha=0.5)

        if T0:
            DeltaJ.append([res[1],res2[1],res3[1]])
            errJ.append([err[1],err2[1],err3[1]])
        else:
            DeltaT.append([res[1],res2[1],res3[1]])
            errT.append([err[1],err2[1],err3[1]])

        if alpha == 1.5 and not T0:
            box = patches.Rectangle(
                (tab[0, 0] ** 2 - 300, abs(tab[0, 1]) - 400),
                2000,
                1000,
                edgecolor="red",
                fill=False,
            )
            ax.add_patch(box)
            ax.text(
                tab[0, 0] ** 2 + 1800,
                abs(tab[0, 1]),
                r"System not extensive",
                color="black",
                fontsize=12,
            )

    p = Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        alpha=0.5,
        label=r"\propto N$",
    )
    plots.append(p)
    labels.append(r" $\propto N$")
    ax.set_xscale("log")
    ax.set_yscale("log")
    FS3.legend(ax, plots, labels, legendLabelBreak=3, loc=(0.02, 0.98, 0.2))
    # fig.savefig("Energy_scaling_%s.pdf"%name, bbox_inches="tight")
    #############################################################################

    labels = [r"$T > 0$", r" $T = 0$", r"$\Delta = 0$"]

fig, ax = FS3.figure(xlabel=r"$\alpha$", ylabel=r"$\Delta$")

for i in [0,1,2]:
    p1 = ax.errorbar(
        np.array(parameters.SELECTPARAMS_T.alpha_list),
        (np.array(DeltaT)[:,i] - 1.0),
        yerr=np.array(errT)[:,i],
        **FS3.errorbarStyle(0),
        ls="",
    )
    p2 = ax.errorbar(
        np.array(parameters.SELECTPARAMS_J.alpha_list),
        (np.array(DeltaJ)[:,i] - 1.0),
        yerr=np.array(errJ)[:,i],
        **FS3.errorbarStyle(1),
        ls="",
    )

p3 = ax.hlines(0.0, 1.4, 5.1, color="black", ls="--", alpha=0.5)
FS3.legend(ax, [p1, p2, p3], labels, loc=(0.75, 0.8, 0.18))
# fig.savefig("Delta_vs_alpha.pdf", bbox_inches="tight")

<Axes: xlabel='$\\alpha$', ylabel='$\\Delta$'>

## Weighted vs Unweighted, Examples of wrong Fit/omega, susceptibility, maybe FS behaviour??, (A LOT TO DO)

# Ewald summation (Done)

## Validation of $\zeta$ function

In [70]:
plt.close("all")

L_space = [32, 64, 128, 256]
FS3.setStyle(L_space)
fig, ax = FS3.figure(xlabel=r"$n$", ylabel=r"$J_n(r)-r^{-\alpha}$")

dx = 1
alpha = 5.0

fig.suptitle(r"$\alpha = %.1f$" % alpha)

nmax = 10000
xmax = nmax * 2
nrange = np.empty(shape=(0), dtype=float)
for digit in [1, 2, 3, 4]:
    nrange = np.hstack(
        (nrange, np.linspace(10 ** (digit - 1), 10 ** (digit), 9, endpoint=False))
    )

nrange = np.hstack((nrange, nmax))

J_ewald_1 = np.zeros_like(nrange, dtype=float)
J_ewald_Lover2 = np.zeros_like(nrange, dtype=float)

Delta_J_ewald_1 = np.zeros_like(nrange, dtype=float)
Delta_J_ewald_Lover2 = np.zeros_like(nrange, dtype=float)

plots = []
labels = []

for L in L_space:
    for i in range(len(nrange)):
        dx = 1
        J_ewald_1[i] = ewald_summation_1D_cutoff(L, dx, alpha, nrange[i])

        dx = L // 2
        J_ewald_Lover2[i] = ewald_summation_1D_cutoff(L, dx, alpha, nrange[i])

    (p,) = ax.plot(nrange, J_ewald_1 - 1.0, **FS3.plotStyle(L), ls="-")
    ax.plot(
        xmax,
        ewald_summation_1D_zeta(L, 1, alpha) - 1.0,
        marker="*",
        ms=12,
        color=p.get_color(),
        clip_on=False,
        zorder=5,
    )
    ax.plot(nrange, J_ewald_Lover2 - (L // 2) ** (-alpha), **FS3.plotStyle(L), ls="-.")
    ax.plot(
        xmax,
        ewald_summation_1D_zeta(L, L // 2, alpha) - ((L // 2) ** (-alpha)),
        marker="^",
        ms=12,
        color=p.get_color(),
        clip_on=False,
        zorder=5,
    )
    plots.append(p)
    labels.append(r"$L = %s$" % L)

plots.extend(
    [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            label=r"$r = 1.0$",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-.",
            label=r"$r = L/2$",
        ),
    ]
)
labels.extend([r"$r = 1$", r"$r = L/2$"])
FS3.legend(ax, plots, labels, legendLabelBreak=4, loc=(0.55, 0.9, 0.2))

ax.set_xlim((1, xmax))
ax.set_xscale("log")

#fig.savefig("zeta_validation_alpha%.1f.pdf" % alpha, bbox_inches="tight")

In [71]:
plt.close("all")

L_space = [32, 64, 128, 256]
FS3.setStyle(L_space)
fig, ax = FS3.figure(xlabel=r"$n$", ylabel=r"$\Delta J$")


dx = 1
alpha = 5.0

fig.suptitle(r"$\alpha = %.1f$"%alpha)

nmax = 10000
xmax = nmax * 2
nrange = np.empty(shape=(0), dtype=float)
for digit in [1, 2, 3, 4]:
    nrange = np.hstack(
        (nrange, np.linspace(10 ** (digit - 1), 10 ** (digit), 9, endpoint=False))
    )

nrange = np.hstack((nrange, nmax))

J_ewald_1 = np.zeros_like(nrange, dtype=float)
J_ewald_Lover2 = np.zeros_like(nrange, dtype=float)

Delta_J_ewald_1 = np.zeros_like(nrange, dtype=float)
Delta_J_ewald_Lover2 = np.zeros_like(nrange, dtype=float)

plots = []
labels = []

for L in L_space:
    for i in range(len(nrange)):
        dx = 1
        J_ewald_1[i] = ewald_summation_1D_cutoff(L, dx, alpha, nrange[i])

        dx = L // 2
        J_ewald_Lover2[i] = ewald_summation_1D_cutoff(L, dx, alpha, nrange[i])

    (p,) = ax.plot(
        nrange,
        ewald_summation_1D_zeta(L, 1, alpha) - J_ewald_1,
        **FS3.plotStyle(L),
        ls="-"
    )
    ax.plot(
        nrange,
        ewald_summation_1D_zeta(L, L // 2, alpha) - J_ewald_Lover2,
        **FS3.plotStyle(L),
        ls="-."
    )
    plots.append(p)
    labels.append(r"$L = %s$" % L)

plots.extend(
    [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            label=r"$r = 1.0$",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-.",
            label=r"$r = L/2$",
        ),
    ]
)
labels.extend([r"$r = 1$", r"$r = L/2$"])
FS3.legend(ax, plots, labels, legendLabelBreak=4, loc=(0.55, 0.98, 0.2))

ax.set_xlim((1, xmax))
ax.set_xscale("log")
ax.set_yscale("log")

#fig.savefig("zeta_correction_alpha%.1f.pdf" % alpha, bbox_inches="tight")

## Compare to SLAC_hopping for $\alpha = 2$

In [79]:
plt.close("all")

L_space = [32, 64, 128, 256]
FS3.setStyle(L_space)
fig, ax = FS3.figure(xlabel=r"$n$", ylabel=r"$\Delta J$")

r = np.arange(0,32)

t = hopping_SLAC_fermions(32,r,0)

ax.plot(-(t**2))
ax.plot(1/r)
ax.plot(ewald_summation_1D_zeta(32,r,2.0))



/tmp/ipykernel_5996/1315328012.py:121: RuntimeWarning: divide by zero encountered in divide
  fx = np.where(x == 0, 0, 1j * (-1)**x * np.pi / (L * np.sin(np.pi * x / L)))
/tmp/ipykernel_5996/1315328012.py:121: RuntimeWarning: invalid value encountered in divide
  fx = np.where(x == 0, 0, 1j * (-1)**x * np.pi / (L * np.sin(np.pi * x / L)))
/tmp/ipykernel_5996/1315328012.py:122: RuntimeWarning: divide by zero encountered in scalar divide
  fy = np.where(y == 0, 0, 1j * (-1)**y * np.pi / (L * np.sin(np.pi * y / L)))
/tmp/ipykernel_5996/1315328012.py:122: RuntimeWarning: invalid value encountered in scalar divide
  fy = np.where(y == 0, 0, 1j * (-1)**y * np.pi / (L * np.sin(np.pi * y / L)))
/tmp/ipykernel_5996/2406361695.py:12: RuntimeWarning: divide by zero encountered in divide
  ax.plot(1/r)


## Total correction

In [4]:
plt.close("all")

L_space = np.array([8, 16, 32, 64, 128, 256])
FS3.setStyle()
fig, ax = FS3.figure(xlabel=r"$L$", ylabel=r"$J_{\zeta}(r)-r^{-\alpha}$")

dxOne = 1.0
dxLover2 = L_space // 2

plots = []
labels = []
i = 0
for alpha in [1.5, 2.5, 5.0]:
    dx = dxOne
    J_simple = dx ** (-alpha)
    p = ax.errorbar(
        L_space,
        ewald_summation_1D_zeta(L_space, dx, alpha) - J_simple,
        **FS3.errorbarStyle(i),
        ls="-"
    )

    plots.append(p)
    labels.append(r"$\alpha = %.1f$" % alpha)

    dx = dxLover2
    J_simple = dx ** (-alpha)
    p = ax.errorbar(
        L_space,
        ewald_summation_1D_zeta(L_space, dx, alpha) - J_simple,
        **FS3.errorbarStyle(i),
        ls="--"
    )
    i += 1

plots.extend(
    [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            label=r"$r = 1.0$",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-.",
            label=r"$r = L/2$",
        ),
    ]
)
labels.extend([r"$r = 1$", r"$r = L/2$"])
FS3.legend(ax, plots, labels, legendLabelBreak=3, loc=(0.02, 0.3, 0.18))
ax.set_xscale("log",base=2)
ax.set_yscale("log")

#fig.savefig("ewald_vs_L.pdf", bbox_inches="tight")

NameError: name 'plt' is not defined

In [70]:
plt.close("all")

L_space = np.array([8, 16, 32, 64, 128, 256])
FS3.setStyle()
fig, ax = FS3.figure(xlabel=r"$L$", ylabel=r"$J_{n}(r)-r^{-\alpha}$")

cutoff = 1000

resOne = np.zeros_like(L_space,dtype=float)
resLover2 = np.zeros_like(L_space,dtype=float)

plots = []
labels = []
a = 0
for alpha in [1.5, 2.5, 5.0]:
    for i in range(len(L_space)):
        resOne[i] = ewald_summation_2D(L_space[i], 1, 0, alpha, cutoff) - 1.0

        dxLover2 = L_space[i]//2
        J_simpleLover2 = dxLover2 **(-alpha)
        resLover2[i] = ewald_summation_2D(L_space[i], dxLover2, 0, alpha, cutoff) - J_simpleLover2

    p = ax.errorbar(
        L_space,
        resOne,
        **FS3.errorbarStyle(a),
        ls="-"
    )

    plots.append(p)
    labels.append(r"$\alpha = %.1f$" % alpha)

    p = ax.errorbar(
        L_space,
        resLover2,
        **FS3.errorbarStyle(a),
        ls="--"
    )
    a += 1

plots.extend(
    [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            label=r"$r = 1.0$",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-.",
            label=r"$r = L/2$",
        ),
    ]
)
labels.extend([r"$r = 1$", r"$r = L/2$"])
FS3.legend(ax, plots, labels, legendLabelBreak=3, loc=(0.02, 0.3, 0.18))
ax.set_xscale("log",base=2)
ax.set_yscale("log")

fig.savefig("ewald_vs_L_FC.pdf", bbox_inches="tight")

KeyboardInterrupt: 

## J(r) with and without Ewald

In [76]:
plt.close("all")

FS3.setStyle()
fig, ax = FS3.figure(xlabel=r"$r$", ylabel=r"$J(r)$")

plots = []
labels = []

alpha=2.0

for L in [32]:
    r_space = np.linspace(-L//2,L//2,L)
    J_simple = r_space**(-alpha)
    J_ewald = ewald_summation_1D_zeta(L, r_space, alpha)
    p = ax.plot(
        r_space,
        J_ewald,
        **FS3.plotStyle(i),
        ls="-"
    )

    ax.plot(
            r_space,
            J_simple,
            **FS3.plotStyle(i),
            ls="-."
        )

    plots.append(p)
    labels.append(r"$\alpha = %.1f$" % alpha)

plots.extend(
    [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            label=r"$r = 1.0$",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-.",
            label=r"$r = L/2$",
        ),
    ]
)
labels.extend([r"$r = 1$", r"$r = L/2$"])
FS3.legend(ax, plots, labels, legendLabelBreak=3, loc=(0.02, 0.3, 0.18))
#ax.set_xscale("log",base=2)
#ax.set_yscale("log")

/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:676: UserWarning: Legend does not support handles for list instances.
A proxy artist may be used instead.
See: https://matplotlib.org/stable/users/explain/axes/legend_guide.html#controlling-the-legend-entries
  ax.legend(


<Axes: xlabel='$r$', ylabel='$J(r)$'>

# LSWT (TO DO)

## Magnon Dispersion (TO DO)

In [ ]:
def powerlaw(x, a, k):
    return k * x**a


def linear_fit(x, a, b):
    return a * x + b


def lambda_k(k, L, alpha, J_perp, connection):
    lamb = 0.0

    match connection:
        case "SR":
            lamb += 2 * np.cos(k[0]) + 2 * np.cos(k[1])
        case "SLAC":
            for x in np.arange(1, L // 2 + 1):
                lamb += (
                    2
                    * ewald_summation_1D_zeta(L, x, alpha)
                    * (np.cos(x * k[0]) + np.cos(x * k[1]))
                )
        case "FC":
            for x in np.arange(1, L // 2 + 1):
                for y in np.arange(1, L // 2 + 1):
                    lamb += (
                        4
                        * 1
                        / np.sqrt(x**2 + y**2) ** (0.5 * alpha)
                        * (np.cos(x * k[0] + y * k[1]))
                    )

    return 0.5 * (lamb + J_perp)


def epsilon_k(L, alpha, J_perp, connection):
    eps = 0.0

    match connection:
        case "SR":
            eps += 4.0
        case "SLAC":
            for x in np.arange(1, L // 2 + 1):
                eps += 4 * ewald_summation_1D_zeta(L, x, alpha)
        case "FC":
            for x in np.arange(1, L // 2 + 1):
                for y in np.arange(1, L // 2 + 1):
                    eps += 4 * 1 / np.sqrt(x**2 + y**2) ** (0.5 * alpha)

    return 0.5 * (eps + J_perp)


def epsilon_0(L, alpha, J_perp, connection):
    return -1.5 * epsilon_k(L, alpha, J_perp, connection)


def energy(k, L, alpha, J_perp, connection):
    eps0 = epsilon_0(L, alpha, J_perp, connection)
    epsk = epsilon_k(L, alpha, J_perp, connection)
    lamb = lambda_k(k, L, alpha, J_perp, connection)
    return np.sqrt(epsk**2 - lamb**2)# +eps0


L = 512
alpha = 2.0
J_perp = 0  # 5.8
periods = 1

k_x = np.linspace(-periods * np.pi, periods * np.pi, periods * L)
k_y = np.linspace(-periods * np.pi, periods * np.pi, periods * L)
kX, kY = np.meshgrid(k_x, k_y)

k = np.linspace(-np.pi * periods, np.pi * periods, L * periods)

ESLAC = energy((kX, kY), L, alpha, J_perp, "SLAC")
ESR = energy((kX, kY), L, alpha, J_perp, "SR")
EFC = energy((kX, kY), L, alpha, J_perp, "FC")

fig1 = plt.figure(figsize=(14, 6), constrained_layout=True)
ax1 = fig1.add_subplot(projection="3d")

ax1.plot_surface(kX, kY, ESLAC, alpha=0.9, color="blue", label="SLAC")
ax1.plot_surface(kX, kY, ESR, alpha=0.9, color="red", label="short-range")

fig2 = plt.figure(figsize=(14, 6), constrained_layout=True)
ax2 = fig2.add_subplot(projection="3d")

ax2.plot_surface(kX, kY, ESLAC, alpha=0.9, color="blue", label="SLAC")
ax2.plot_surface(kX, kY, EFC, alpha=0.9, color="green", label="fully-connected")

for ax in [ax1, ax2]:
    ax.set_xlim(-periods * np.pi, periods * np.pi)
    ax.set_ylim(-periods * np.pi, periods * np.pi)
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.set_zlabel(r"$E(\mathbf{k})$")

    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_xticklabels([r"$-\pi$", r"$0$", r"$\pi$"])
    ax.set_yticks([-np.pi, 0, np.pi])
    ax.set_yticklabels([r"$-\pi$", r"$0$", r"$\pi$"])

    ax.set_zticks([0,1,2,3])
    ax.set_zticklabels([0, 1,2,3])
    ax.legend(loc="upper right")
    ax.set_box_aspect((1,1,0.7))
    ax.view_init(elev=30, azim=-60) 

fig1.savefig("Dispersion_Magnon_SLAC_SR.pdf")
fig2.savefig("Dispersion_Magnon_SLAC_FC.pdf")

## $z: E \propto k^{-z}$ (TO DO)

# Single Fits + Histograms (TO DO)

## $U_2(g,L)=f((g/gc-1) L^{1/\nu})(1+cL^{-\omega})$ (DONE)

### Single fit (DONE)

In [31]:
plt.close("all")

T0 = False
fixed_gc_nu = False

if T0:
    select_params = parameters.SELECTPARAMS_J
    loc_params = parameters.SINGLEFITPARAMS_BINDER_J
    name = "J"
    histResults = parameters.HISTRESULTS_J
else:
    select_params = parameters.SELECTPARAMS_T
    loc_params = parameters.SINGLEFITPARAMS_BINDER_T
    name = "T"
    histResults = parameters.HISTRESULTS_T

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
# select_params = dataclasses.replace(select_params, LRange=LRange)


for index in range(len(select_params.alpha_list))[0:1]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="binder",
        plot=False,
    )

    if fixed_gc_nu:
        gc_nu = {
            "gc": histResults.gc_binder[index, 0],
            "nu": histResults.nu_binder[index, 0],
        }
        start_params = [2.3]
    else:
        gc_nu = None
        start_params = [
            select_params.gc_estimate[index],
            select_params.nu_estimate[index],
            2.0
            #0.475,104,1.24,-1.06,1.96,-8.5,25,-26
        ]

    fitFunc = cr.fitfunc(
        fstring="f((x/gc-1)*L**(1/nu))*(1+c*L**(-omega))",
        vars=["gc", "nu", "omega", "c"],
        fixed=gc_nu,
        polyOrder=3,
    )

    fit_res = cr.fit_data(
        data,
        fitFunc,
        start_params=start_params,
        fitSummary=True,
        full_output=True,
    )

    params, dparams, redChi2, mesg, ierr, cov = fit_res
    var = fitFunc.unpack(params)
    dvar = fitFunc.unpack(dparams)

    corr = cov / np.outer(dparams, dparams)
    # print(pd.DataFrame(corr, index=fitFunc.free_vars, columns=fitFunc.free_vars).round(3))
    # print()

    fig, ax = cr.plot_fit(
        data,
        fitFunc,
        params,
        xlabel=cl_runs.obs_labels["g_param"],
        ylabel=r"$U_2$",
        loc_legend=loc_params.loc_legend[index],
    )

    if fixed_gc_nu:
        fig.suptitle(
            r"$\alpha = %.2f, %s_c = %.3f, \nu = %.3f$"
            % (alpha, name, histResults.gc_binder[index, 0], histResults.nu_binder[index, 0])
        )
    else:
        fig.suptitle(
                    r"$\alpha = %.2f$, free parameters"
                    % (alpha)
                )

    dataCollapse, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="binder",
        xfunc="(x/gc - 1)*L**(1/nu)",
        yfunc="y/(1 + c*L**(-omega))",
        arg=var,
    )

    ax.vlines(
        var["gc"],
        ax.get_ylim()[0],
        ax.get_ylim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )
    ax.hlines(
        var["a0"],
        ax.get_xlim()[0],
        ax.get_xlim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    inset, ax_inset = cr.plot_DataCollapse(
        dataCollapse,
        fitFunc,
        fit_res,
        xlabel=r"$\tau L^{\frac{1}{\nu}}$",
        ylabel=r"$U_2/(1+cL^{-\omega})$",
        insetFigure=fig,
        loc=loc_params.loc_inset[index],
    )

    ax_inset.vlines(
        0,
        ax_inset.get_ylim()[0],
        ax_inset.get_ylim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    ax_inset.hlines(
        var["a0"],
        ax_inset.get_xlim()[0],
        ax_inset.get_xlim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    # if fixed_gc_nu:
    #     fig.savefig("alpha%.2f_U2_%s_fixed.pdf" % (alpha, name),bbox_inches="tight")
    #     with open("alpha%.2f_U2_%s_fixed.txt"% (alpha, name), "w") as text_file:
    #         text_file.write(FS3.fitSummary(fitFunc,fit_res,silent=True))
    # else:
    #     fig.savefig("alpha%.2f_U2_%s_free.pdf" % (alpha, name),bbox_inches="tight")
    #     with open("alpha%.2f_U2_%s_free.txt"% (alpha, name), "w") as text_file:
    #         text_file.write(FS3.fitSummary(fitFunc,fit_res,silent=True))

chi2/d.o.f = 2564.08       
gc = 2.3309(9)
nu = 1.045(2)
omega = 7.74(7)
c = 1462(14)
a0 = 1.20(1)
a1 = -0.626(7)
a2 = 0.132(1)
a3 = -0.00805(8)



### create Histogram (DONE!)

In [ ]:
plt.close("all")

T0 = True
n = 1

if T0:
    select_params = parameters.SELECTPARAMS_J
    name = "J"
else:
    select_params = parameters.SELECTPARAMS_T
    name = "T"

LRange = select_params.LRange.copy()
LRange[1] = [64, 128]
#select_params = dataclasses.replace(select_params, LRange=LRange)

fitFunc = cr.fitfunc(
    fstring="f((x/gc-1)*L**(1/nu))*(1+c*L**(-omega))",
    vars=["gc", "nu", "omega", "c"],
    polyOrder=3,
)

for index in range(len(select_params.alpha_list))[2:]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="binder",
        plot=False,
    )

    params_list = []
    dparams_list = []
    chi_list = []

    print("alpha = %.2f"%alpha)
    for i in range(n):
        if i%(int(n/100))==0:
            print("process: %d"%int(i/n*100))

        start_params = rand_start_params([select_params.gc_estimate[index], select_params.nu_estimate[index]],fitFunc)
        fit_res = cr.fit_data(
            cr.resample(data),
            fitFunc,
            start_params=start_params,
            fitSummary=False,
        )

        params, dparams, redChi2, mesg, ierr = fit_res
        params_list.append(params)
        dparams_list.append(dparams)
        chi_list.append(redChi2)


        
    hist_data = {}
    hist_data["chi2"] = chi_list

    for i in range(len(fitFunc.free_vars)):
        var_name = fitFunc.free_vars[i]
        err_name = "d" + fitFunc.free_vars[i]
        hist_data[var_name] = np.array(params_list)[:, i]
        hist_data[err_name] = np.array(dparams_list)[:, i]

    hist_df = pd.DataFrame.from_dict(hist_data)
    #hist_df.to_csv("hist_alpha%.2f_U2_%s.csv" % (alpha, name))

alpha = 2.50


ZeroDivisionError: integer modulo by zero

### plot histogram (DONE!)

In [25]:
plt.close("all")

T0 = False

if T0:
    params = parameters.SELECTPARAMS_J
    hist_params = parameters.HISTPARAMS_BINDER_J
    name = "J"
else:
    params = parameters.SELECTPARAMS_T
    hist_params = parameters.HISTPARAMS_BINDER_T
    name = "T"

# x_range = hist_params.x_range.copy()
# y_range = hist_params.y_range.copy()
# x_range[1] = [0.705, 0.9]
# y_range[1] = [5.795, 5.82]
# hist_params = dataclasses.replace(hist_params, x_range=x_range, y_range=y_range)

nu_list = []
gc_list = []

for index in range(len(params.alpha_list))[:1]:
    alpha = params.alpha_list[index]

    hist_df = pd.read_csv(
        "/home/daniel/Master_thesis/DataHist%s/N10000/hist_binder%s_alpha%.2f_unbias.csv"
        % (name, name, alpha)
    )

    # hist_df = pd.read_csv("/home/daniel/Master_thesis/DataHistJ/N10000/hist_binderJ_alpha2.00_LangParams.csv")
    # x_range=[0.71,0.76]
    # y_range=[5.7979,5.7988]

    x_fit, y_fit, x_min, y_min = cr.scatter_histograms(
        xData=hist_df["c"],
        yData=hist_df["omega"],
        x_steps=hist_params.x_steps[index],
        y_steps=hist_params.y_steps[index],
        x_range=hist_params.x_range[index],
        y_range=hist_params.y_range[index],
        # y_steps=0.0002,
        # x_range=x_range,
        # y_range=y_range,
        chi2=hist_df["chi2"],
        num_bins=50,
        #savefig="alpha%.2f_U2_%s.pdf" % (alpha, name),
        xlabel=r"$\nu$",
        ylabel=r"$%s_c$" % name,
    )

    nu_list.append(x_fit)
    gc_list.append(y_fit)

print("nu_binder = np.array([", end="")
for nu in nu_list:
    print("[%f,%f]" % (nu[0], nu[1]), end=",")
print("])")

print("gc_binder = np.array([", end="")
for gc in gc_list:
    print("[%f,%f]" % (gc[0], gc[1]), end=",")
print("])")

/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/numpy/lib/_histograms_impl.py:1080: RuntimeWarning: invalid value encountered in divide
  hist /= s
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/numpy/lib/_histograms_impl.py:897: RuntimeWarning: invalid value encountered in divide
  return n / db / n.sum(), bin_edges


ValueError: array must not contain infs or NaNs

## $\rho_s(g,L)=L^{-z}f((g/gc-1) L^{1/\nu})(1+cL^{-\omega})$ (Done)

### Single Fit (Done)

In [111]:
plt.close("all")

T0 = False
fixed_gc_nu_z = True

if T0:
    select_params = parameters.SELECTPARAMS_J
    loc_params = parameters.SINGLEFITPARAMS_SPIN_STIFFNESS_J
    name = "J"
    histResults = parameters.HISTRESULTS_J
else:
    select_params = parameters.SELECTPARAMS_T
    loc_params = parameters.SINGLEFITPARAMS_SPIN_STIFFNESS_T
    name = "T"
    histResults = parameters.HISTRESULTS_T

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
# select_params = dataclasses.replace(select_params, LRange=LRange)


for index in range(len(select_params.alpha_list))[4:5]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="spin_stiffness",
        plot=False,
    )

    if fixed_gc_nu_z:
        gc_nu_z = {
            "gc": histResults.gc_spin_stiffness[index, 0],
            "nu": histResults.nu_spin_stiffness[index, 0],
            "z": histResults.z_spin_stiffness[index, 0],
        }
        start_params = [2.0]
    else:
        gc_nu_z = None
        start_params = [
            select_params.z_estimate[index],
            select_params.gc_estimate[index],
            select_params.nu_estimate[index],
            5.0
        ]

    fitFunc = cr.fitfunc(
        fstring="f((x/gc-1)*L**(1/nu))*L**(-z)*(1+c*L**(-omega))",
        vars=["z","gc", "nu", "omega", "c"],
        fixed=gc_nu_z,
        polyOrder=3,
    )

    fit_res = cr.fit_data(
        data,
        fitFunc,
        start_params=start_params,
        fitSummary=True,
        full_output=True,
    )

    params, dparams, redChi2, mesg, ierr, cov = fit_res
    var = fitFunc.unpack(params)
    dvar = fitFunc.unpack(dparams)

    corr = cov / np.outer(dparams, dparams)
    # print(pd.DataFrame(corr, index=fitFunc.free_vars, columns=fitFunc.free_vars).round(3))
    # print()

    data, _, _ = cl_runs.get_x_vs_y_data(
            xname="g_param",
            yname="spin_stiffness",
            yfunc="y*L**z",
            arg = var,
            plot=False,
        )

    params_no_z = np.copy(params)
    if not fixed_gc_nu_z:
        params_no_z[0] = 0.0
        fitFunc_no_z = fitFunc
    else:
        # z is baked into fitFunc's lambda as a fixed symbol, so it can't be
        # zeroed out via params like the free case - rebuild with z fixed to 0 instead
        fitFunc_no_z = cr.fitfunc(
            fstring=fitFunc.fstring,
            vars=["z", "gc", "nu", "omega", "c"],
            fixed={**gc_nu_z, "z": 0.0},
            polyOrder=fitFunc.polyOrder,
        )

    fig, ax = cr.plot_fit(
        data,
        fitFunc_no_z,
        params_no_z,
        xlabel=cl_runs.obs_labels["g_param"],
        ylabel=r"$\rho_s L^z$",
        loc_legend=loc_params.loc_legend[index],
    )

    if fixed_gc_nu_z:
        fig.suptitle(
            r"$\alpha = %.2f, %s_c = %.3f, \nu = %.3f, z = %.3f$"
            % (alpha, name, histResults.gc_spin_stiffness[index, 0], histResults.nu_spin_stiffness[index, 0], histResults.z_spin_stiffness[index, 0])
        )
    else:
        fig.suptitle(
                    r"$\alpha = %.2f$, free parameters"
                    % (alpha)
                )

    dataCollapse, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="spin_stiffness",
        xfunc="(x/gc - 1)*L**(1/nu)",
        yfunc="y*L**(z)/(1 + c*L**(-omega))",
        arg=var,
    )

    ax.vlines(
        var["gc"],
        ax.get_ylim()[0],
        ax.get_ylim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )
    ax.hlines(
        var["a0"],
        ax.get_xlim()[0],
        ax.get_xlim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    inset, ax_inset = cr.plot_DataCollapse(
        dataCollapse,
        fitFunc,
        fit_res,
        xlabel=r"$\tau L^{\frac{1}{\nu}}$",
        ylabel=r"$\rho_sL^{z}/(1+cL^{-\omega})$",
        insetFigure=fig,
        loc=loc_params.loc_inset[index],
    )

    ax_inset.vlines(
        0,
        ax_inset.get_ylim()[0],
        ax_inset.get_ylim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    ax_inset.hlines(
        var["a0"],
        ax_inset.get_xlim()[0],
        ax_inset.get_xlim()[1],
        colors="k",
        linestyles="-",
        alpha=0.2,
    )

    if fixed_gc_nu_z:
        fig.savefig("alpha%.2f_rho_%s_fixed.pdf" % (alpha, name),bbox_inches="tight")
        with open("alpha%.2f_rho_%s_fixed.txt"% (alpha, name), "w") as text_file:
            text_file.write(FS3.fitSummary(fitFunc,fit_res,silent=True))
    else:
        fig.savefig("alpha%.2f_rho_%s_free.pdf" % (alpha, name),bbox_inches="tight")
        with open("alpha%.2f_rho_%s_free.txt"% (alpha, name), "w") as text_file:
            text_file.write(FS3.fitSummary(fitFunc,fit_res,silent=True))

chi2/d.o.f = 3.96          
omega = 3.2(4)
c = -36(38)
a0 = 0.41995(9)
a1 = -0.610(3)
a2 = -0.39(2)
a3 = 0.26(5)
gc = 0.8(8)
nu = 2(2)
z = -0.01(-1)



/tmp/ipykernel_9998/3270750646.py:172: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("alpha%.2f_rho_%s_fixed.pdf" % (alpha, name),bbox_inches="tight")


### create Histogram (DONE TAKE OLD ONES)

In [ ]:
plt.close("all")

T0 = True
n = 10

if T0:
    select_params = parameters.SELECTPARAMS_J
    name = "J"
else:
    select_params = parameters.SELECTPARAMS_T
    name = "T"

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
#select_params = dataclasses.replace(select_params, LRange=LRange)

fitFunc = cr.fitfunc(
    fstring="f((x/gc-1)*L**(1/nu))*L**(-z)*(1+c*L**(-omega))",
    vars=["z","gc", "nu", "omega", "c"],
    polyOrder=3,
)

for index in range(len(select_params.alpha_list))[:]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="spin_stiffness",
        plot=False,
    )

    params_list = []
    dparams_list = []
    chi_list = []

    print("alpha = %.2f"%alpha)
    for i in range(n):
        # if i%(int(n/100))==0:
        #     print("process: %d"%int(i/n*100))

        start_params = rand_start_params([select_params.z_estimate[index],select_params.gc_estimate[index], select_params.nu_estimate[index]],fitFunc)
        fit_res = cr.fit_data(
            cr.resample(data),
            fitFunc,
            start_params=start_params,
            fitSummary=False,
        )

        params, dparams, redChi2, mesg, ierr = fit_res
        params_list.append(params)
        dparams_list.append(dparams)
        chi_list.append(redChi2)
        
    hist_data = {}
    hist_data["chi2"] = chi_list

    for i in range(len(fitFunc.free_vars)):
        var_name = fitFunc.free_vars[i]
        err_name = "d" + fitFunc.free_vars[i]
        hist_data[var_name] = np.array(params_list)[:, i]
        hist_data[err_name] = np.array(dparams_list)[:, i]

    hist_df = pd.DataFrame.from_dict(hist_data)
    hist_df.to_csv("hist_alpha%.2f_rho_%s.csv" % (alpha, name))

alpha = 1.50
alpha = 2.00
alpha = 2.50
alpha = 3.00
alpha = 3.50
alpha = 4.00
alpha = 4.50
alpha = 5.00


/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


### Plot Histogram (Done)

#### $\nu(g_c)$ (DONE)

In [115]:
plt.close("all")

T0 = True

if T0:
    params = parameters.SELECTPARAMS_J
    hist_params = parameters.HISTPARAMS_SPIN_STIFFNESS_NU_J
    name = "J"
else:
    params = parameters.SELECTPARAMS_T
    hist_params = parameters.HISTPARAMS_SPIN_STIFFNESS_NU_T
    name = "T"

nu_list = []
gc_list = []

for index in range(len(params.alpha_list))[:]:
    alpha = params.alpha_list[index]

    hist_df = pd.read_csv(
        "/home/daniel/Master_thesis/DataHist%s/N10000/hist_spin_stiffness%s_alpha%.2f_unbias.csv"
        % (name, name, alpha)
    )

    x_fit, y_fit, x_min, y_min = cr.scatter_histograms(
        xData=hist_df["nu"],
        yData=hist_df["gc"],
        x_steps=hist_params.x_steps[index],
        y_steps=hist_params.y_steps[index],
        x_range=hist_params.x_range[index],
        y_range=hist_params.y_range[index],
        # y_steps=0.0002,
        # x_range=x_range,
        # y_range=y_range,
        chi2=hist_df["chi2"],
        num_bins=50,
        savefig="alpha%.2f_rho_nu_%s.pdf" % (alpha, name),
        xlabel=r"$\nu$",
        ylabel=r"$%s_c$" % name,
    )

    nu_list.append(x_fit)
    gc_list.append(y_fit)

print("nu_rho = np.array([", end="")
for nu in nu_list:
    print("[%f,%f]" % (nu[0], nu[1]), end=",")
print("])")

print("gc_rho = np.array([", end="")
for gc in gc_list:
    print("[%f,%f]" % (gc[0], gc[1]), end=",")
print("])")

/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.p

nu_rho = np.array([[0.782428,0.002731],[0.773378,0.004708],[0.728809,0.003447],[0.714038,0.003166],[0.720367,0.002668],[0.717652,0.002074],[0.718202,0.004409],[0.723739,0.006952],])
gc_rho = np.array([[9.980339,0.000174],[5.794953,0.000058],[4.329596,0.000037],[3.594004,0.000034],[3.182863,0.000030],[2.942135,0.000028],[2.795722,0.000058],[2.703874,0.000051],])


#### $z(\omega)$ (Done)

In [71]:
plt.close("all")

T0 = False

if T0:
    params = parameters.SELECTPARAMS_J
    hist_params = parameters.HISTPARAMS_SPIN_STIFFNESS_Z_J
    name = "J"
else:
    params = parameters.SELECTPARAMS_T
    hist_params = parameters.HISTPARAMS_SPIN_STIFFNESS_Z_T
    name = "T"

z_list = []
omega_list = []

for index in range(len(params.alpha_list))[:]:
    alpha = params.alpha_list[index]

    hist_df = pd.read_csv(
            "/home/daniel/Master_thesis/DataHist%s/N10000/hist_spin_stiffness%s_alpha%.2f_unbias.csv"
            % (name, name, alpha)
        )

    x_fit, y_fit, x_min, y_min = cr.scatter_histograms(
        xData=hist_df["z"],
        yData=hist_df["omega"],
        x_steps=hist_params.x_steps[index],
        y_steps=hist_params.y_steps[index],
        x_range=hist_params.x_range[index],
        y_range=hist_params.y_range[index],
        # y_steps=0.0002,
        # x_range=x_range,
        # y_range=y_range,
        chi2=hist_df["chi2"],
        num_bins=50,
        savefig="alpha%.2f_rho_z_%s.pdf" % (alpha, name),
        xlabel=r"$z$",
        ylabel=r"$\omega$",
    )

    z_list.append(x_fit)
    omega_list.append(y_fit)

print("z_rho = np.array([", end="")
for z in z_list:
    print("[%f,%f]" % (z[0], z[1]), end=",")
print("])")

print("omega_rho = np.array([", end="")
for omega in omega_list:
    print("[%f,%f]" % (omega[0], omega[1]), end=",")
print("])")

/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.p

z_rho = np.array([[-0.596622,0.001168],[-0.139009,0.001429],[-0.028231,0.001478],[-0.000192,0.001068],[-0.010823,0.001176],[-0.096016,0.008759],])
omega_rho = np.array([[-0.002969,0.099676],[0.000668,0.102189],[0.001246,0.099034],[-0.001402,0.099753],[-0.001713,0.100912],[0.002351,0.096375],])


## $\rho_s(U_2,L)=L^{-z}f(U_2)(1+cL^{-\omega})$ (DONE)

### Single fits (Done)

In [159]:
plt.close("all")

T0 = False
fixed_z = True

if T0:
    select_params = parameters.SELECTPARAMS_J
    loc_params = parameters.SINGLEFITPARAMS_RHO_U2_J
    name = "J"
    histResults = parameters.HISTRESULTS_J
else:
    select_params = parameters.SELECTPARAMS_T
    loc_params = parameters.SINGLEFITPARAMS_RHO_U2_T
    name = "T"
    histResults = parameters.HISTRESULTS_T

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
# select_params = dataclasses.replace(select_params, LRange=LRange)


for index in range(len(select_params.alpha_list))[4:5]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="spin_stiffness",
        plot=False,
    )

    if fixed_z:
        z = {
            "z": histResults.z_rho_U2[index, 0],
        }
        start_params = [2.0]
    else:
        z = None
        start_params = [
            select_params.z_estimate[index],
            2.0,
        ]

    fitFunc = cr.fitfunc(
        fstring="f(x)*L**(-z)*(1+c*L**(-omega))",
        vars=["z", "omega", "c"],
        fixed=z,
        polyOrder=3,
    )

    fit_res = cr.fit_data(
        data,
        fitFunc,
        start_params=start_params,
        fitSummary=True,
        full_output=True,
    )

    params, dparams, redChi2, mesg, ierr, cov = fit_res
    var = fitFunc.unpack(params)
    dvar = fitFunc.unpack(dparams)

    corr = cov / np.outer(dparams, dparams)
    # print(pd.DataFrame(corr, index=fitFunc.free_vars, columns=fitFunc.free_vars).round(3))
    # print()

    fig, ax = cr.plot_fit(
        data,
        fitFunc,
        params,
        xlabel=r"$U_2$",
        ylabel=r"$\rho_s$",
        loc_legend=loc_params.loc_legend[index],
    )

    if fixed_z:
        fig.suptitle(
            r"$\alpha = %.2f, z = %.3f$"
            % (alpha, histResults.z_rho_U2[index, 0])
        )
    else:
        fig.suptitle(
                    r"$\alpha = %.2f$, free parameters"
                    % (alpha)
                )

    dataCollapse, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="spin_stiffness",
        xfunc="x",
        yfunc="y*L**(z)/(1 + c*L**(-omega))",
        arg=var,
    )

    inset, ax_inset = cr.plot_DataCollapse(
        dataCollapse,
        fitFunc,
        fit_res,
        xlabel=r"$U_2$",
        ylabel=r"$\rho_sL^{z}/(1+cL^{-\omega})$",
        insetFigure=fig,
        loc=loc_params.loc_inset[index],
    )

    if fixed_z:
        fig.savefig("alpha%.2f_rho_U2_%s_fixed.pdf" % (alpha, name), bbox_inches="tight")
        with open("alpha%.2f_rho_U2_%s_fixed.txt" % (alpha, name), "w") as text_file:
            text_file.write(FS3.fitSummary(fitFunc, fit_res, silent=True))
    else:
        fig.savefig("alpha%.2f_rho_U2_%s_free.pdf" % (alpha, name), bbox_inches="tight")
        with open("alpha%.2f_rho_U2_%s_free.txt" % (alpha, name), "w") as text_file:
            text_file.write(FS3.fitSummary(fitFunc, fit_res, silent=True))

chi2/d.o.f = 15.17         
omega = 0.487(7)
c = 3.03(5)
a0 = -50.1376(6)
a1 = 174.106(2)
a2 = -201.827(2)
a3 = 78.448(5)
z = 0.0001(1)



/tmp/ipykernel_9998/4185452704.py:111: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("alpha%.2f_rho_U2_%s_fixed.pdf" % (alpha, name), bbox_inches="tight")


### create Histograms (Done Take given)

In [6]:
plt.close("all")

T0 = False
n = 10000

if T0:
    select_params = parameters.SELECTPARAMS_J
    name = "J"
else:
    select_params = parameters.SELECTPARAMS_T
    name = "T"

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
#select_params = dataclasses.replace(select_params, LRange=LRange)

fitFunc = cr.fitfunc(
    fstring="f(x)*L**(-z)*(1+c*L**(-omega))",
    vars=["z", "omega", "c"],
    polyOrder=3,
)

for index in range(len(select_params.alpha_list))[2:3]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="spin_stiffness",
        plot=False,
    )

    params_list = []
    dparams_list = []
    chi_list = []

    print("alpha = %.2f"%alpha)
    for i in range(n):
        if i%(int(n/100))==0:
            print("process: %d"%int(i/n*100))

        start_params = rand_start_params([select_params.z_estimate[index],2.0],fitFunc)
        fit_res = cr.fit_data(
            cr.resample(data),
            fitFunc,
            start_params=start_params,
            fitSummary=False,
        )

        params, dparams, redChi2, mesg, ierr = fit_res
        params_list.append(params)
        dparams_list.append(dparams)
        chi_list.append(redChi2)
        
    hist_data = {}
    hist_data["chi2"] = chi_list

    for i in range(len(fitFunc.free_vars)):
        var_name = fitFunc.free_vars[i]
        err_name = "d" + fitFunc.free_vars[i]
        hist_data[var_name] = np.array(params_list)[:, i]
        hist_data[err_name] = np.array(dparams_list)[:, i]

    hist_df = pd.DataFrame.from_dict(hist_data)
    hist_df.to_csv("hist_alpha%.2f_rho_U2_%s.csv" % (alpha, name))

alpha = 2.25
process: 0


/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:687: RuntimeWarning: overflow encountered in divide
  df_dx = [delf / delx for delf, delx in zip(df, dx)]
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_linesearch.py:89: RuntimeWarning: invalid value encountered in dot
  return np.dot(gval[0], pk)


process: 1


/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


process: 2


...

process: 96


/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


process: 97
process: 98


<lambdifygenerated-5>:2: RuntimeWarning: overflow encountered in scalar power
  return L**(-z)*(1 + L**(-omega)*c)*(a0 + a1*x + a2*x**2 + a3*x**3)
/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]
/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
<lambdifygenerated-5>:2: RuntimeWarning: overflow encountered in scalar power
  return L**(-z)*(1 + L**(-omega)*c)*(a0 + a1*x + a2*x**2 + a3*x**3)
/home/daniel/Master_thesis/Python_analysis_tools/FS3.py:151: RuntimeWarning: overflow encountered in square
  return np.sum(np.array(res) ** 2)
/home/daniel/Master_thesis/Python_analysis_tools/.venv/lib/py

process: 99


### Plot Histograms (DONE)

In [ ]:
plt.close("all")

T0 = True

if T0:
    params = parameters.SELECTPARAMS_J
    hist_params = parameters.HISTPARAMS_RHO_U2_Z_J
    name = "J"
else:
    params = parameters.SELECTPARAMS_T
    hist_params = parameters.HISTPARAMS_RHO_U2_Z_T
    name = "T"

fitFunc = cr.fitfunc(
    fstring="f(x)*L**(-z)*(1+c*L**(-omega))",
    vars=["z", "omega", "c"],
    polyOrder=3,
)

z_list = []
omega_list = []

for index in range(len(params.alpha_list))[:5]:
    alpha = params.alpha_list[index]

    # hist_df = pd.read_csv(
    #         "/home/daniel/Master_thesis/DataHist%s/N10000/hist_z%s_alpha%.2f_unbias.csv"
    #         % (name, name, alpha)
    #     )

    hist_df = pd.read_csv(
                "/home/daniel/Master_thesis/DataHist%s/N10000/hist_alpha%.2f_rho_U2_%s.csv"
                % (name, alpha, name)
            )

    hist_df_correct = hist_df.copy()
    mask = hist_df_correct["omega"] < 0
    hist_df_correct.loc[mask,"z"] = hist_df["z"][mask] + hist_df["omega"][mask]
    hist_df_correct.loc[mask,"omega"] = -1.0*hist_df["omega"][mask]

    x_fit, y_fit, x_min, y_min = cr.scatter_histograms(
        xData=hist_df_correct["z"],
        yData=hist_df_correct["omega"],
        x_steps=hist_params.x_steps[index],
        y_steps=hist_params.y_steps[index],
        x_range=hist_params.x_range[index],
        y_range=hist_params.y_range[index],
        # y_steps=0.0002,
        # x_range=x_range,
        # y_range=y_range,
        chi2=hist_df_correct["chi2"],
        num_bins=50,
        #savefig="alpha%.2f_rho_U_2_%s.pdf" % (alpha, name),
        xlabel=r"$z$",
        ylabel=r"$\omega$",
    )

    z_list.append(x_fit)
    omega_list.append(y_fit)

print("z_rho_U2 = np.array([", end="")
for z in z_list:
    print("[%f,%f]" % (z[0], z[1]), end=",")
print("])")

print("omega_rho_U2 = np.array([", end="")
for omega in omega_list:
    print("[%f,%f]" % (omega[0], omega[1]), end=",")
print("])")

/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:899: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("hist_" + savefig, bbox_inches="tight")
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:899: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("hist_" + savefig, bbox_inches="tight")
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:899: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("hist_" + savefig, bbox_inches="tight")
/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:899: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("hist_" + savefig, bbox_inches="tight")
/home/daniel/Master_thesis/Python_analysis_t

z_rho_U2 = np.array([[-0.077003,0.038253],[0.400293,0.004286],[0.765522,0.004634],[0.953320,0.006430],[1.017063,0.007105],])
omega_rho_U2 = np.array([[0.374497,0.126081],[0.168993,0.006912],[0.355659,0.012126],[0.956073,0.053577],[0.916612,0.072524],])


## $m^2(U_2,L)=L^{-2\beta/\nu}f(U_2)(1+cL^{-\omega})$(TO DO)

### Single Fits(To do: Fixed)

In [54]:
plt.close("all")

T0 = False
fixed_twoBetaNuInv = False

if T0:
    select_params = parameters.SELECTPARAMS_J
    loc_params = parameters.SINGLEFITPARAMS_BETA_J
    name = "J"
    histResults = parameters.HISTRESULTS_J
else:
    select_params = parameters.SELECTPARAMS_T
    loc_params = parameters.SINGLEFITPARAMS_BETA_T
    name = "T"
    histResults = parameters.HISTRESULTS_T

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
# select_params = dataclasses.replace(select_params, LRange=LRange)


for index in range(len(select_params.alpha_list))[1:2]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="mag2",
        yfunc="y*L**(-4)",
        plot=False,
    )

    if fixed_twoBetaNuInv:
        twoBetaNuInv = {
            "twoBetaNuInv": histResults.beta_mag2_U[index, 0],
        }
        start_params = [2.0]
    else:
        twoBetaNuInv = None
        start_params = [
            select_params.twoBetaNuInv_estimate[index],
            2.0,
        ]

    fitFunc = cr.fitfunc(
        fstring="f(x)*L**(-twoBetaNuInv)*(1+c*L**(-omega))",
        vars=["twoBetaNuInv", "omega", "c"],
        fixed=twoBetaNuInv,
        polyOrder=3,
    )

    fit_res = cr.fit_data(
        data,
        fitFunc,
        start_params=start_params,
        fitSummary=True,
        full_output=True,
    )

    params, dparams, redChi2, mesg, ierr, cov = fit_res
    var = fitFunc.unpack(params)
    dvar = fitFunc.unpack(dparams)

    print("2 beta/nu = %.3f --> beta = %.3f, chi2 = %.5f" % (var["twoBetaNuInv"],var["twoBetaNuInv"] / 2 * histResults.nu_binder[index, 0], redChi2))

    corr = cov / np.outer(dparams, dparams)
    # print(pd.DataFrame(corr, index=fitFunc.free_vars, columns=fitFunc.free_vars).round(3))
    # print()

    fig, ax = cr.plot_fit(
        data,
        fitFunc,
        params,
        xlabel=r"$U_2$",
        ylabel=r"$m^2$",
        loc_legend=loc_params.loc_legend[index],
    )

    if fixed_twoBetaNuInv:
        fig.suptitle(
            r"$\alpha = %.2f, 2\beta/\nu = %.3f$"
            % (alpha, name, histResults.beta_mag2_U[index, 0])
        )
    else:
        fig.suptitle(
                    r"$\alpha = %.2f$, free parameters"
                    % (alpha)
                )

    dataCollapse, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="mag2",
        xfunc="x",
        yfunc="y*L**(twoBetaNuInv-4)/(1 + c*L**(-omega))",
        arg=var,
    )

    inset, ax_inset = cr.plot_DataCollapse(
        dataCollapse,
        fitFunc,
        fit_res,
        xlabel=r"$U_2$",
        ylabel=r"$m^2L^{2\beta/\nu}/C_L$",
        insetFigure=fig,
        loc=loc_params.loc_inset[index],
    )

    # if fixed_twoBetaNuInv:
    #     fig.savefig("alpha%.2f_beta_%s_fixed.pdf" % (alpha, name), bbox_inches="tight")
    #     with open("alpha%.2f_beta_%s_fixed.txt" % (alpha, name), "w") as text_file:
    #         text_file.write(FS3.fitSummary(fitFunc, fit_res, silent=True))
    # else:
    #     fig.savefig("alpha%.2f_beta_%s_free.pdf" % (alpha, name), bbox_inches="tight")
    #     with open("alpha%.2f_beta_%s_free.txt" % (alpha, name), "w") as text_file:
    #         text_file.write(FS3.fitSummary(fitFunc, fit_res, silent=True))

chi2/d.o.f = 0.00          
twoBetaNuInv = 0.9(1)
omega = 1.1(6)
c = -3(15)
a0 = -0.0(9)
a1 = 2(6)
a2 = -2(13)
a3 = 2(9)

2 beta/nu = 0.903 --> beta = 0.450, chi2 = 0.00043


### create Histograms (Done)

In [ ]:
plt.close("all")

T0 = False
n = 100

if T0:
    select_params = parameters.SELECTPARAMS_J
    name = "J"
else:
    select_params = parameters.SELECTPARAMS_T
    name = "T"

# LRange = select_params.LRange.copy()
# LRange[1] = [64, 128]
# select_params = dataclasses.replace(select_params, LRange=LRange)

fitFunc = cr.fitfunc(
    fstring="f(x)*L**(-twoBetaNuInv)*(1+c*L**(-omega))",
    vars=["twoBetaNuInv", "omega", "c"],
    polyOrder=3,
)

for index in range(len(select_params.alpha_list))[1:2]:
    alpha = select_params.alpha_list[index]
    cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=select_params.LRange[index],
        gRange=select_params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="binder",
        yname="mag2",
        yfunc="y*L**(-4)",
        plot=False,
    )

    params_list = []
    dparams_list = []
    chi_list = []

    print("alpha = %.2f" % alpha)
    for i in range(n):
        if i%(int(n/100))==0:
            print("process: %d"%int(i/n*100))

        start_params = rand_start_params(
            [select_params.twoBetaNuInv_estimate[index], 2], fitFunc
        )
        fit_res = cr.fit_data(
            cr.resample(data),
            fitFunc,
            start_params=start_params,
            fitSummary=False,
        )

        params, dparams, redChi2, mesg, ierr = fit_res
        params_list.append(params)
        dparams_list.append(dparams)
        chi_list.append(redChi2)

    hist_data = {}
    hist_data["chi2"] = chi_list

    for i in range(len(fitFunc.free_vars)):
        var_name = fitFunc.free_vars[i]
        err_name = "d" + fitFunc.free_vars[i]
        hist_data[var_name] = np.array(params_list)[:, i]
        hist_data[err_name] = np.array(dparams_list)[:, i]

    hist_df = pd.DataFrame.from_dict(hist_data)
    hist_df.to_csv("hist_alpha%.2f_beta_%s.csv" % (alpha, name))

alpha = 1.50
process: 0
process: 1
process: 2
process: 3
process: 4


KeyboardInterrupt: 

### plot Histograms (DONE)

In [40]:
plt.close("all")

T0 = False

if T0:
    params = parameters.SELECTPARAMS_J
    hist_params = parameters.HISTPARAMS_BETA_J
    name = "J"
else:
    params = parameters.SELECTPARAMS_T
    hist_params = parameters.HISTPARAMS_BETA_T
    name = "T"

twoBetaNuInv_list = []
omega_list = []

for index in range(len(params.alpha_list))[:1]:
    alpha = params.alpha_list[index]

    # hist_df = pd.read_csv(
    #         "/home/daniel/Master_thesis/DataHist%s/N10000/hist_beta%s_alpha%.2f_unbias.csv"
    #         % (name, name, alpha)
    #     )

    hist_df = pd.read_csv("/home/daniel/Master_thesis/DataHist%s/N10000/hist_alpha%.2f_beta_%s.csv" % (name,alpha, name))

    hist_df_correct = hist_df.copy()
    mask = hist_df_correct["omega"] < 0
    hist_df_correct.loc[mask,"twoBetaNuInv"] = hist_df["twoBetaNuInv"][mask] + hist_df["omega"][mask]
    hist_df_correct.loc[mask,"omega"] = -1.0*hist_df["omega"][mask]


    x_fit, y_fit, x_min, y_min = cr.scatter_histograms(
        xData=hist_df_correct["twoBetaNuInv"],
        yData=hist_df_correct["omega"],
        x_steps=hist_params.x_steps[index],
        y_steps=hist_params.y_steps[index],
        x_range=hist_params.x_range[index],
        y_range=hist_params.y_range[index],
        # y_steps=0.0002,
        # x_range=x_range,
        # y_range=y_range,
        chi2=hist_df_correct["chi2"],
        num_bins=50,
        savefig="alpha%.2f_beta_%s.pdf" % (alpha, name),
        xlabel=r"$2\beta/\nu$",
        ylabel=r"$\omega$",
    )

    twoBetaNuInv_list.append(x_fit)
    omega_list.append(y_fit)

print("twoBetaNuInv_mag2_U = np.array([", end="")
for twoBetaNuInv in twoBetaNuInv_list:
    print("[%f,%f]" % (twoBetaNuInv[0], twoBetaNuInv[1]), end=",")
print("])")

print("omega_mag2 = np.array([", end="")
for omega in omega_list:
    print("[%f,%f]" % (omega[0], omega[1]), end=",")
print("])")

/home/daniel/Master_thesis/Python_analysis_tools/Collected_runs.py:900: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("hist_" + savefig, bbox_inches="tight")


twoBetaNuInv_mag2_U = np.array([[1.000636,0.004498],])
omega_mag2 = np.array([[0.931335,0.066518],])


## Beta Different Fit functions (DONE)

In [5]:
FS3.setStyle()

T0 = True
mag_density = True

if mag_density:
    Lm = 0.0
    yfunc = "y/(Ns**2)"
    ylabel = r"$m^2$"
else:
    Lm = 4.0
    yfunc = "y"
    ylabel = r"$M^2$"

if T0:
    select_params = parameters.SELECTPARAMS_J
    name = "J"
else:
    select_params = parameters.SELECTPARAMS_T
    name = "T"

fitFunc1 = cr.fitfunc(
    fstring="f(x)*L**(-kappa)",
    # fstring="f(x/(1+c*L**(-omega)))*L**(-kappa)",
    vars=["kappa", "omega", "c"],
    polyOrder=3,
)

fitFunc2 = cr.fitfunc(
    # fstring="f(x)*L**(-kappa)",
    fstring="f(x/(1+c*L**(-omega)))*L**(-kappa)",
    vars=["kappa", "omega", "c"],
    polyOrder=3,
)

fitFunc3 = cr.fitfunc(
    # fstring="f(x)*L**(-kappa)",
    fstring="f(x)*L**(-kappa)*(1+c*L**(-omega))",
    vars=["kappa", "omega", "c"],
    polyOrder=3,
)

fig, ax = FS3.figure(xlabel=r"$\alpha$",ylabel=r"$\beta$")

for n in range(3):
    beta1 = []
    beta2 = []
    beta3 = []
    corr = []

    for index in range(len(select_params.alpha_list))[0:]:

        alpha = select_params.alpha_list[index]
        cl_runs = cr.collected_runs(select_params.pathRuns[index], T0, cutoff=100)

        cl_runs.select(
            LRange=select_params.LRange[index],
            gRange=select_params.gRange[index],
        )

        data, _, _ = cl_runs.get_x_vs_y_data(
            xname="binder",
            yname="mag2",
            yfunc=yfunc,
            plot=False,
        )

        fit_res1 = cr.fit_data(
            data,
            fitFunc1,
            start_params=[select_params.TwoBetaNuInv_estimate[index] - Lm],
            fitSummary=False,
        )

        params1, dparams1, redChi21, mesg, ierr = fit_res1
        var1 = fitFunc1.unpack(params1)
        dvar1 = fitFunc1.unpack(dparams1)

        fit_res2 = cr.fit_data(
            data,
            fitFunc2,
            start_params=[select_params.TwoBetaNuInv_estimate[index] - Lm],
            fitSummary=False,
        )
        params2, dparams2, redChi22, mesg, ierr = fit_res2
        var2 = fitFunc2.unpack(params2)
        dvar2 = fitFunc2.unpack(dparams2)

        fit_res3 = cr.fit_data(
            data,
            fitFunc3,
            start_params=[select_params.TwoBetaNuInv_estimate[index] - Lm, 1.0],
            fitSummary=False,
        )
        params3, dparams3, redChi23, mesg, ierr = fit_res3
        var3 = fitFunc3.unpack(params3)
        dvar3 = fitFunc3.unpack(dparams3)

        print(params2[0] + Lm)

        # print("raw beta(%.2f) = %.4f, chi2= %.3f" % (alpha,(params1[0] + 4) * select_params.nu_estimate[index]*0.5, redChi21))
        # print("corrected beta(%.2f) = %.4f, chi2= %.3f" % (alpha,(params2[0] + 4) * select_params.nu_estimate[index]*0.5,redChi22))

        beta1.append((params1[0] + Lm) * nuJ_binder_bias[index][0] * 0.5)
        beta2.append((params2[0] + Lm) * nuJ_binder_bias[index][0] * 0.5)
        beta3.append((params3[0] + Lm) * nuJ_binder_bias[index][0] * 0.5)
        corr.append((params3[0] + params3[1] + Lm) * nuJ_binder_bias[index][0] * 0.5)

        # fig, ax = cr.plot_fit(
        #     data,
        #     fitFunc1,
        #     params1,
        #     xlabel=r"$U_2$",
        #     ylabel=ylabel,
        # )

        # dataCollapse1, _, _ = cl_runs.get_x_vs_y_data(
        #     xname="binder",
        #     yname="mag2",
        #     xfunc="x",
        #     yfunc="y*L**(kappa)",
        #     arg=var1,
        # )

        # inset, ax_inset = cr.plot_DataCollapse(
        #     dataCollapse1,
        #     fitFunc1,
        #     fit_res1,
        #     xlabel=r"$U_2$",
        #     ylabel=r"$m^2L^{2\beta/\nu}$",
        #     insetFigure=fig,
        #     loc=3,
        # )

        # fig.savefig("beta_nocorr_a%.2f.pdf"%(alpha))

        # fig, ax = cr.plot_fit(
        #     data,
        #     fitFunc2,
        #     params2,
        #     xlabel=r"$U_2$",
        #     ylabel=ylabel",
        # )

        # dataCollapse2, _, _ = cl_runs.get_x_vs_y_data(
        #     xname="binder",
        #     yname="mag2",
        #     xfunc="x/(1+c*L**(-omega))",
        #     yfunc="y*L**(kappa)",
        #     arg=var2,
        # )

        # inset, ax_inset = cr.plot_DataCollapse(
        #     dataCollapse2,
        #     fitFunc2,
        #     fit_res2,
        #     xlabel=r"$U_2(1+cL^{-\omega})$",
        #     ylabel=r"$m^2L^{2\beta/\nu}$",
        #     insetFigure=fig,
        #     loc=3,
        # )

        # fig.savefig("beta_Ucorr_a%.2f.pdf"%(alpha))

        # fig, ax = cr.plot_fit(
        #     data,
        #     fitFunc3,
        #     params3,
        #     xlabel=r"$U_2$",
        #     ylabel=ylabel,
        # )

        # dataCollapse3, _, _ = cl_runs.get_x_vs_y_data(
        #     xname="binder",
        #     yname="mag2",
        #     xfunc="x",
        #     yfunc="y*L**(kappa)/(1+c*L**(-omega))",
        #     arg=var3,
        # )

        # inset, ax_inset = cr.plot_DataCollapse(
        #     dataCollapse3,
        #     fitFunc3,
        #     fit_res3,
        #     xlabel=r"$U_2$",
        #     ylabel=r"$m^2L^{2\beta/\nu}/(1+cL^{-\omega})$",
        #     insetFigure=fig,
        #     loc=3,
        # )

        # fig.savefig("beta_Mcorr_a%.2f.pdf"%(alpha))

    ax.errorbar(
        select_params.alpha_list,
        beta1,
        alpha=1.0,
        **FS3.errorbarStyle(0),
        label=ylabel + r"$=L^{-\kappa}f(U_2)$"
    )
    ax.errorbar(
        select_params.alpha_list,
        beta2,
        alpha=1.0,
        **FS3.errorbarStyle(3),
        label=ylabel + r"$=L^{-\kappa}f(U_2/(1+cL^{-\omega}))$"
    )
    ax.errorbar(
        select_params.alpha_list,
        beta3,
        alpha=1.0,
        **FS3.errorbarStyle(5),
        label=ylabel + r"$=L^{-\kappa}f(U_2)(1+cL^{-\omega})$"
    )
    # plt.plot(select_params.alpha_list, corr, color="orange", alpha=0.5)

ax.errorbar( betaSong24[:, 0] - 1, betaSong24[:, 1], yerr=betaSong24[:, 2], color="black", label="Song")
plots = [
    ax.errorbar([], [], **FS3.errorbarStyle(0)),
    ax.errorbar([], [], **FS3.errorbarStyle(3)),
    ax.errorbar([], [], **FS3.errorbarStyle(5)),
    ax.errorbar([], [], yerr=[], color="black"),
]

labels = [
    (ylabel + r"$=L^{-\kappa}f(U_2)$"),
    (ylabel + r"$=L^{-\kappa}f(U_2/(1+cL^{-\omega}))$"),
    (ylabel + r"$=L^{-\kappa}f(U_2)(1+cL^{-\omega})$"),
    r"Song at ($\alpha + 1$)",
]

FS3.legend(ax,plots,labels,loc=(0.5,0.7,0.18),legendLabelBreak=4)

fig.savefig("m_different_beta_fitfuncs.pdf",bbox_inches="tight")


#     start_params = [
#         dic_start_params["gc"][index],
#         dic_start_params["nu"][index],
#         dic_start_params["z"][index],
#         # dic_start_params["c"][index],
#         # dic_start_params["omega"][index],
#         # dic_start_params["a0"][index],
#         # dic_start_params["a1"][index],
#         # dic_start_params["a2"][index],
#         # dic_start_params["a3"][index],
#     ]

#     params_list = []
#     dparams_list = []
#     chi_list = []

#     for i in range(1):
#         if i % 1000 == 0:
#             print(alpha, i)

#         fit_res = cr.fit_data(
#             cr.resample(data),
#             fitFunc,
#             start_params=rand_start_params(start_params, fitFunc),
#             fitSummary=False,
#         )

#         params, dparams, redChi2, mesg, ierr = fit_res
#         params_list.append(params)
#         dparams_list.append(dparams)
#         chi_list.append(redChi2)

#     hist_data = {}
#     hist_data["chi2"] = chi_list

#     for i in range(len(fitFunc.vars)):
#         var_name = fitFunc.vars[i]
#         err_name = "d" + fitFunc.vars[i]
#         hist_data[var_name] = np.array(params_list)[:, i]
#         hist_data[err_name] = np.array(dparams_list)[:, i]

#     hist_df = pd.DataFrame.from_dict(hist_data)
#     #hist_df.to_csv("hist_spin_stiffness%s_alpha%.2f_unbias.csv" % (name, alpha))

# plt.close("all")

1.301397835324451
1.2286118217679929
1.2236003556625814
1.1014606588162899
1.0482716316488663
1.037396508439285
1.0537915306438483
1.0257076359514157
1.3014423302892633
1.263198023231134
1.165449029645587
1.1012193255594094
1.0471910309461636
1.0363824708456897
1.0411935381088482
1.0146073721810998
1.3329884896909383
1.3076784829135895
1.185120533944444
1.1010430846512982
1.05998894217713
1.0266556531730493
1.0116704639126624
1.0069201528941523


# Crit. Exponents vs. $\alpha$ (TO DO)

## $g_c(\alpha)$ (Done)

In [97]:
plt.close("all")

T0 = False

if T0:
    params = parameters.SELECTPARAMS_J
    hist_results = parameters.HISTRESULTS_J
    name = "J"
    ylabel = r"$J^{\perp}_c$"
    SR_limit = 2.522
    MF_limit = None
    dataCompare = parameters.LITVALUES_J.gc
    dataCompare = [dataCompare - np.ones((len(dataCompare),3))*[1,0,0],r"Song at $(\alpha + 1)$"]
else:
    params = parameters.SELECTPARAMS_T
    params = dataclasses.replace(params, alpha_list=params.alpha_list[:-1])
    hist_results = parameters.HISTRESULTS_T
    name = "T"
    ylabel = r"$T_c$"
    SR_limit = 0.0
    MF_limit = None
    dataCompare = parameters.LITVALUES_T.gc
    dataCompare = [dataCompare - np.ones((len(dataCompare),3))*[1,0,0],r"Zhao at $(\alpha + 1)$"]


data1 = [np.column_stack((params.alpha_list, hist_results.gc_binder)), r"$U_2$"]
data2 = [
    np.column_stack((params.alpha_list, hist_results.gc_spin_stiffness)),
    r"$\rho_s$",
]

data_array = [data1, data2,dataCompare]

fig, ax = cr.plot_exp_vs_alpha(
    data_array, ylabel, invert=False, SR_limit=SR_limit, MF_limit=MF_limit, loc_legend="best",savefig="gc_%s"%name
)



## $\nu(\alpha)$ (Done)

In [101]:
plt.close("all")

T0 = True

if T0:
    params = parameters.SELECTPARAMS_J
    hist_results = parameters.HISTRESULTS_J
    name = "J"
    ylabel = r"$\nu^{\prime}$"
    SR_limit = 0.7106
    MF_limit = 0.75
    dataCompare = parameters.LITVALUES_J.nu
    dataCompare = [
        dataCompare - np.ones((len(dataCompare), 3)) * [1, 0, 0],
        r"Song at $(\alpha + 1)$",
    ]
else:
    params = parameters.SELECTPARAMS_T
    params = dataclasses.replace(params, alpha_list=params.alpha_list[:-1])
    hist_results = parameters.HISTRESULTS_T
    name = "T"
    ylabel = r"$\nu^{\prime}$"
    SR_limit = None
    MF_limit = 1.0
    dataCompare = parameters.LITVALUES_T.nu
    dataCompare = [
        dataCompare - np.ones((len(dataCompare), 3)) * [1, 0, 0],
        r"Zhao at $(\alpha + 1)$",
    ]


data1 = [np.column_stack((params.alpha_list, hist_results.nu_binder)), r"$U_2$"]
data2 = [
    np.column_stack((params.alpha_list, hist_results.nu_spin_stiffness)),
    r"$\rho_s$",
]

data_array = [data1, data2, dataCompare]

fig, ax = cr.plot_exp_vs_alpha(
    data_array,
    ylabel,
    invert=False,
    SR_limit=SR_limit,
    MF_limit=MF_limit,
    loc_legend="best",
    # savefig="nu_%s"%name
)

if T0:

    ax.errorbar(
        parameters.nuJ_Lang_alpha2[0],
        parameters.nuJ_Lang_alpha2[1],
        parameters.nuJ_Lang_alpha2[2],
        **FS3.errorbarStyle(7)
    )

    box1 = patches.Rectangle(
        (
            parameters.nuJ_Lang_alpha2[0] - 0.1,
            parameters.nuJ_Lang_alpha2[1] - parameters.nuJ_Lang_alpha2[2],
        ),
        0.2,
        2 * parameters.nuJ_Lang_alpha2[2],
        edgecolor=FS3.errorbarStyle(7)["color"],
        fill=False,
    )
    ax.add_patch(box1)
    ax.annotate(
        r"only $L \geq 64$",
        (parameters.nuJ_Lang_alpha2[0], parameters.nuJ_Lang_alpha2[1]),
        (3.0,0.76),
        arrowprops=dict(arrowstyle="->",color=FS3.errorbarStyle(7)["color"]),
        color="black",
        fontsize=15,
    )

    box2 = patches.Rectangle(
        (
            2.0 - 0.1,
            hist_results.nu_binder[1,0] - hist_results.nu_binder[1,1],
        ),
        0.2,
        2 * parameters.nuJ_Lang_alpha2[2],
        edgecolor=FS3.errorbarStyle(0)["color"],
        fill=False,
    )
    ax.add_patch(box2)
    ax.annotate(
        r"all $L$",
        (2.0,hist_results.nu_binder[1,0]),
        (3.0,0.74),
        arrowprops=dict(arrowstyle="->",color=FS3.errorbarStyle(0)["color"]),
        color="black",
        fontsize=15,
    )

fig.savefig("nu_%s_vs_alpha.pdf"%name)

## $z(\alpha)$ (T= DO)

### via $\rho(g = gc, L)$ (Only for T = 0) (DONE)

In [78]:
plt.close("all")


def powerlaw(x, a, z):
    return a * x ** (-z)


T0 = True

if T0:
    params = parameters.SELECTPARAMS_J
    hist_results = parameters.HISTRESULTS_J
    name = "J"
    L_range = np.linspace(24, 196, 100)
    xticks = [32, 48, 64, 96, 128]
    xticklabels = ["32", "48", "64", "96", "128"]
    ylim = [1e-3, 1e0]

    start_params = [
        [0.2458231897782261, -0.2103078936079375],
        [0.3322293693636365, 0.24857844903378878],
        [0.599647130680161, 0.6276889318745448],
        [1.1163672508004718, 0.9263391481680466],
        [1.128859219525521, 0.9978707748185301],
        [0.872558995410391, 0.9642664127918702],
        [0.8775208906354435, 0.9940346078704997],
        [0.8249062369647915, 0.9916586588411185],
    ]
else:
    params = parameters.SELECTPARAMS_T
    hist_results = parameters.HISTRESULTS_T
    name = "T"
    fitfunc = powerlaw
    L_range = np.linspace(48, 384, 100)
    xticks = [64, 128, 256]
    xticklabels = ["64", "128", "256"]

    ylim = [1e-1, 1e1]

    start_params = [
        [0.18357156363639132, -0.5709864707814714],
        [0.40413703809697876, -0.07521542881733395],
        [0.4387780145321402, 0.007355138849951037],
        [0.44081123339282335, 0.022774728409319406],
        [0.4865269067479497, 0.03705344531128993],
    ]

FS3.setStyle(Llist=np.sort(list(params.alpha_list), axis=0))
fig = plt.figure(figsize=(7.7, 5.6))
grid_space = mpl.gridspec.GridSpec(
    1, 2, wspace=0.1, hspace=0.1, width_ratios=[0.66, 0.33], figure=fig
)

ax0 = fig.add_subplot(grid_space[0])
axLegend = fig.add_subplot(grid_space[1])
axLegend.axis("off")

ax0.set_xscale("log")
ax0.set_yscale("log")
ax0.set_xlabel(r"$L$")
ax0.set_ylabel(r"$\rho_s(\tau = 0)$")

ax0.set_xlim(L_range[0], L_range[-1])
ax0.set_ylim(*ylim)
ax0.set_xticks(xticks)
ax0.set_xticklabels(xticklabels)
ax0.xaxis.set_minor_locator(NullLocator())

plots = []
labels = []

for index in range(len(params.alpha_list))[:]:
    alpha = params.alpha_list[index]
    cl_runs = cr.collected_runs(params.pathRuns[index], T0, cutoff=100)

    cl_runs.select(
        LRange=params.LRange[index],
        gRange=params.gRange[index],
    )

    data, _, _ = cl_runs.get_x_vs_y_data(
        xname="g_param",
        yname="spin_stiffness",
        plot=False,
    )

    g0 = hist_results.gc_binder[index, 0]

    tab = np.empty((0, 3))
    for L in np.sort(list(data.keys()), axis=0):
        imin = np.argmin(np.abs(data[L][:, 0] - g0))
        tab = np.vstack((tab, np.array([L, data[L][imin, 1], data[L][imin, 2]])))

    p = ax0.errorbar(tab[:, 0], tab[:, 1], tab[:, 2], ls="", **FS3.errorbarStyle(alpha))
    res, cov = sp.optimize.curve_fit(
        fitfunc, tab[:, 0], tab[:, 1], p0=start_params[index][:2]
    )

    ax0.plot(L_range, fitfunc(L_range, *res), **FS3.plotStyle(alpha))
    plots.append(p)
    labels.append(r"$\alpha = %.2f: z = %s$" % (alpha, FS3.errText(res[1], cov[1, 1] ** 0.5)))

FS3.legend(axLegend, plots, labels, legendLabelBreak=8, loc=(-0.25, 1.0, 0.2))
fig.savefig("z_%s_rho_vs_L.pdf" % name)

/tmp/ipykernel_5902/1555960852.py:104: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.savefig("z_%s_rho_vs_L.pdf" % name)


### via Histograms and LSWT

In [112]:
plt.close("all")

T0 = True

if T0:
    params = parameters.SELECTPARAMS_J
    hist_results = parameters.HISTRESULTS_J
    name = "J"
    ylabel = r"$z$"
    SR_limit = 1.0
    MF_limit = None
else:
    params = parameters.SELECTPARAMS_T
    params = dataclasses.replace(params, alpha_list=params.alpha_list[:-1])
    hist_results = parameters.HISTRESULTS_T
    name = "T"
    ylabel = r"$z$"
    SR_limit = 0.0
    MF_limit = None


data1 = [np.column_stack((params.alpha_list, hist_results.z_spin_stiffness)), r"$\rho_s(\tau L^{1/\nu})$"]
data2 = [np.column_stack((params.alpha_list, hist_results.z_rho_U2)), r"$\rho_s(U_2)$"]

data_array = [data1, data2,[np.zeros_like(data1[0]),r""]]

print(data_array)

fig, ax = cr.plot_exp_vs_alpha(
    data_array, ylabel, invert=False, SR_limit=SR_limit, MF_limit=MF_limit, loc_legend="best",#savefig="z_%s"%name
)



[[array([[ 1.50000e+00, -2.67237e-01,  8.95000e-04],
       [ 2.00000e+00,  2.29030e-01,  1.22100e-03],
       [ 2.50000e+00,  6.11782e-01,  1.61300e-03],
       [ 3.00000e+00,  8.27408e-01,  2.00000e-03],
       [ 3.50000e+00,  9.34670e-01,  1.98300e-03],
       [ 4.00000e+00,  9.72330e-01,  1.93500e-03],
       [ 4.50000e+00,  9.79337e-01,  3.23400e-03],
       [ 5.00000e+00,  9.83914e-01,  2.91800e-03]]), '$\\rho_s(\\tau L^{1/\\nu})$'], [array([[ 1.500000e+00, -7.700300e-02,  3.825300e-02],
       [ 2.000000e+00,  4.002930e-01,  4.286000e-03],
       [ 2.500000e+00,  7.655220e-01,  4.634000e-03],
       [ 3.000000e+00,  9.533200e-01,  6.430000e-03],
       [ 3.500000e+00,  1.017063e+00,  7.105000e-03],
       [ 4.000000e+00,  1.057453e+00,  5.060000e-03],
       [ 4.500000e+00,  1.010736e+00,  2.248200e-02],
       [ 5.000000e+00,  1.085485e+00,  1.075200e-02]]), '$\\rho_s(U_2)$'], [array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0

## $\beta (\alpha)$ (TO DO)

### via $m^2(g = gc, L)$

### via Histograms (DONE)

In [67]:
plt.close("all")

T0 = True

plot_beta = True #if True, plot beta vs alpha, if False, plot 2 beta/nu vs alpha

if T0:
    params = parameters.SELECTPARAMS_J
    hist_results = parameters.HISTRESULTS_J
    name = "J"
    if plot_beta:
        ylabel = r"$\beta$"
        SR_limit = 0.366
        MF_limit = 0.5
    else:
        ylabel = r"$2\beta/\nu$"
        SR_limit = 2*0.366/0.7106
        MF_limit = 2*0.5/0.75
    dataCompare = parameters.LITVALUES_J.beta
    dataCompare = [
        dataCompare - np.ones((len(dataCompare), 3)) * [1, 0, 0],
        r"Song at $(\alpha + 1)$",
    ]
else:
    params = parameters.SELECTPARAMS_T
    params = dataclasses.replace(params, alpha_list=params.alpha_list[:-1])
    hist_results = parameters.HISTRESULTS_T
    name = "T"
    if plot_beta:
            ylabel = r"$\beta$"
            SR_limit = None
            MF_limit = 0.5
    else:
        ylabel = r"$2\beta/\nu$"
        SR_limit = None
        MF_limit = 1.0
    dataCompare = parameters.LITVALUES_T.beta
    dataCompare = [
        dataCompare - np.ones((len(dataCompare), 3)) * [1, 0, 0],
        r"Zhao at $(\alpha + 1)$",
    ]

data = [np.column_stack((params.alpha_list, hist_results.twoBetaNuInv_mag2_U)), r"$m_s^2(U_2)$"]

if plot_beta:
    nu = np.column_stack((params.alpha_list, hist_results.nu_binder))
    data[0] = parameters.convert_Beta_to_2BetaNuInv(data[0], nu,inverse=True)

    fig_name = "beta_%s"%name
else:
    dataCompare[0] = parameters.convert_Beta_to_2BetaNuInv(dataCompare[0], parameters.LITVALUES_J.nu,inverse=False)

    fig_name = "2beta_nu_%s"%name

data_array = [data, dataCompare]

fig, ax = cr.plot_exp_vs_alpha(
    data_array,
    ylabel,
    invert=False,
    SR_limit=SR_limit,
    MF_limit=MF_limit,
    loc_legend="best",
    savefig=fig_name
)